# 02 · Features & sentiment-scorer validation (Phase 4)

Input: `data/clean/hourly_merged.parquet` (Phase 3, frozen: 664 hours, Sept 8 00:00 → Oct 5 15:00 UTC).
Output: `data/clean/features.parquet`, the **single feature table** used by `04_model.ipynb` and the
Track C dashboard, and `data/validation/sentiment_sample.csv` for hand-labelling.

The logic lives in `processing/features.py` (unit-tested in `tests/test_features.py`). This notebook
runs it and shows the checks.

In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

FIGURES = Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
WINDOW = ("2026-09-08", "2026-10-05T16:00")  # frozen collection window (handoff §2)

## 1. The target: why it is shifted by two hours

`xau_usd` at row *t* is the GC=F price at **t:00**. Row *t*'s sentiment summarises posts from
**t:00 to t:59**, so it is only fully known at **(t+1):00**.

* ❌ `xau_usd[t+1] > xau_usd[t]`: that move happens *while* the posts are written. It measures
  co-movement, and accuracy would look better than anything tradable.
* ✅ `price_direction = xau_usd[t+2] > xau_usd[t+1]`: features known at (t+1):00 predict the
  *next* hour's move. A tie counts as "not up".

A row is `target_valid` only when both prices exist (the market is closed on weekends, during the daily
21:00 UTC break and on US holidays). See pitfall #11.

In [2]:
from processing.features import run, build_features, PRICE_FEATURES, SENTIMENT_FEATURES

features = run()
valid = features[features["target_valid"]]

assert len(features) == 664 and features.index.is_unique
assert (features.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all()
assert len(valid) == 431
assert str(valid.index.min()) == "2026-09-08 03:00:00+00:00"
assert str(valid.index.max()) == "2026-10-05 13:00:00+00:00"
print(f"target-valid rows: {len(valid)}  up: {valid['price_direction'].mean():.1%}  "
      f"-> majority-class baseline ≈ {1 - valid['price_direction'].mean():.1%}")
print("valid rows per ISO week:", valid.groupby(valid.index.isocalendar().week).size().tolist())

Wrote /home/user/GoldPulse/data/clean/features.parquet: 664 rows, 431 target-valid, 48.0% up
target-valid rows: 431  up: 48.0%  -> majority-class baseline ≈ 52.0%
valid rows per ISO week: [85, 110, 112, 110, 14]


## 2. Features

All computed on the **full 664-hour calendar grid** (so "lag 1" never jumps across a weekend), and
all **trailing**: no `center=True`, and no `shift(-k)` anywhere except the target.

| Feature | Definition |
|---|---|
| `sent_roll_mean_3h/6h` | rolling mean of `mean_sentiment` (forward-filled series) |
| `sent_roll_std_3h/6h` | rolling std over **observed** hours only (`mean_sentiment.where(had_posts)`, ≥2 needed). Forward-filled hours repeat a value and would fake low volatility. `_filled` = NaN→0 for the model |
| `*_lag1/2` | `mean_sentiment`, `bullish_ratio`, `post_count` at t-1, t-2 |
| `log_post_count(_lag1/2)` | `log1p(post_count)` |
| `log_engagement` | `log1p(mean_engagement)`, NaN→0, plus `engagement_missing` flag (76 hours have no settled posts) |
| `day_of_week`, `hour_of_day`, `hour_sin/cos` | calendar; gold trading is strongly intraday |
| `last_return` | `xau_usd[t]/xau_usd[t-1] - 1`, the **price-only** feature for the H3 comparison. NaN across closures; `last_return_filled` = NaN→0, plus `return_missing` flag |

In [3]:
model_cols = PRICE_FEATURES + SENTIMENT_FEATURES
print("model input columns:", len(model_cols))
print("NaNs in model inputs on target-valid rows:", int(valid[model_cols].isna().sum().sum()))
print("target-valid rows whose last_return spans a closure:", int(valid["return_missing"].sum()))
valid[model_cols + ["price_direction"]].describe().T.round(3)

model input columns: 19
NaNs in model inputs on target-valid rows: 0
target-valid rows whose last_return spans a closure: 38


,count,mean,std,min,25%,50%,75%,max
last_return_filled,431.0,-0.000,0.003,-0.009,-0.001,0.000,0.001,0.013
hour_sin,431.0,0.092,0.682,-1.000,-0.500,0.259,0.707,1.000
hour_cos,431.0,-0.041,0.726,-1.000,-0.707,-0.259,0.707,1.000
mean_sentiment,431.0,0.305,0.282,-0.688,0.135,0.299,0.496,0.996
sent_roll_mean_3h,431.0,0.301,0.186,-0.628,0.193,0.292,0.427,0.938
sent_roll_mean_6h,431.0,0.298,0.136,-0.279,0.217,0.295,0.383,0.600
sent_roll_std_3h_filled,431.0,0.218,0.146,0.000,0.117,0.191,0.310,0.935
sent_roll_std_6h_filled,431.0,0.253,0.111,0.059,0.180,0.237,0.304,0.730
mean_sentiment_lag1,431.0,0.301,0.286,-0.688,0.118,0.296,0.498,0.996
mean_sentiment_lag2,431.0,0.297,0.287,-0.688,0.117,0.296,0.498,0.996


### No-look-ahead check
If a feature at hour *t* used anything after *t*, recomputing the features on a frame **cut off**
at some hour would change the earlier rows. They must be identical.

In [4]:
hourly = pd.read_parquet("data/clean/hourly_merged.parquet")
for cut in ["2026-09-15 12:00", "2026-09-28 00:00", "2026-10-03 07:00"]:
    full = build_features(hourly)[model_cols].loc[:cut]
    short = build_features(hourly.loc[:cut])[model_cols]
    pd.testing.assert_frame_equal(full, short)
print("features at t are unchanged when all rows after t are removed ✔")

features at t are unchanged when all rows after t are removed ✔


## 3. Sentiment-scorer validation (hand-labelled sample)

The scorer (`sentiment/scorer.py`: VADER + `FINANCE_LEXICON`) is a fixed, rule-based lexicon
scorer, **not a trained model**. Checking it against human labels is how we measure its accuracy.

* Population: posts in the frozen window, de-duplicated with the same rule as Phase 3
  (`processing.bucket.dedupe_posts`).
* Sample: **15 per scorer class** (bullish > +0.05, neutral, bearish < −0.05), `random_state=42`.
  We stratify because otherwise about a third would be neutral and bearish would be rare (283 of 2,419).
  Agreement is therefore reported per class and as a balanced figure, not as a population estimate.
* **A person labels `human_label`** (bullish / neutral / bearish) without looking at the scorer's
  label first, and writes their name in `labelled_by`. The code prepares the file but fills in no labels.
* The CSV is written **only if it doesn't exist**, so re-running this notebook never wipes labels.

In [5]:
from processing.bucket import load_raw, dedupe_posts, NEUTRAL_BAND

def scorer_label(s):
    return np.where(s > NEUTRAL_BAND, "bullish", np.where(s < -NEUTRAL_BAND, "bearish", "neutral"))

posts, _ = load_raw()
posts = posts[(posts["created_utc"] >= WINDOW[0]) & (posts["created_utc"] < WINDOW[1])]
posts = dedupe_posts(posts).assign(scorer_label=lambda d: scorer_label(d["sentiment"]))
print(len(posts), "de-duplicated posts in window")
print(posts["scorer_label"].value_counts().to_string())

SAMPLE = Path("data/validation/sentiment_sample.csv")
if SAMPLE.exists():
    print(f"\n{SAMPLE} already exists, so it is left untouched (it may hold human labels).")
else:
    sample = (posts.groupby("scorer_label", group_keys=False)
                   .sample(n=15, random_state=42)
                   .sample(frac=1, random_state=42))  # shuffle so classes aren't in blocks
    sample = sample[["post_id", "subreddit", "text", "sentiment", "scorer_label"]].assign(
        human_label="", labelled_by="")
    SAMPLE.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(SAMPLE, index=False, encoding="utf-8")
    print(f"\nwrote {SAMPLE} ({len(sample)} posts)")

2419 de-duplicated posts in window
scorer_label
bullish    1339
neutral     797
bearish     283

data/validation/sentiment_sample.csv already exists, so it is left untouched (it may hold human labels).


In [6]:
from sklearn.metrics import confusion_matrix

labels = ["bearish", "neutral", "bullish"]
val = pd.read_csv(SAMPLE, encoding="utf-8", keep_default_na=False)
val["human_label"] = val["human_label"].str.strip().str.lower()
done = val[val["human_label"].isin(labels)]

if done.empty:
    print(f"Awaiting human labels: 0 of {len(val)} rows labelled. Fill human_label in {SAMPLE} "
          "and re-run this notebook.")
else:
    agree = (done["human_label"] == done["scorer_label"])
    who = ", ".join(sorted(set(done["labelled_by"]) - {""})) or "(labeller not recorded)"
    print(f"Labelled: {len(done)} of {len(val)} by {who}")
    print(f"Agreement (scorer sign at ±{NEUTRAL_BAND} vs human): {agree.mean():.1%} ({agree.sum()}/{len(done)})")
    print("Per scorer class:", done.assign(a=agree).groupby("scorer_label")["a"].mean().round(2).to_dict())
    by_sub = done.assign(a=agree).groupby("subreddit")["a"].agg(labelled="size", agreement="mean")
    print("Per subreddit (tiny n, indicative only):")
    print(by_sub.round(2).to_string())
    no_sb = agree[done["subreddit"] != "Silverbugs"]
    print(f"Without r/Silverbugs: {no_sb.mean():.1%} ({no_sb.sum()}/{len(no_sb)})")
    print("Human label share by subreddit:")
    print(pd.crosstab(done["subreddit"], done["human_label"], normalize="index").round(2).to_string())
    cm = pd.DataFrame(confusion_matrix(done["human_label"], done["scorer_label"], labels=labels),
                      index=[f"human {l}" for l in labels], columns=[f"scorer {l}" for l in labels])
    display(cm)
    print("\nDisagreements (look for missing lexicon terms or sarcasm):")
    display(done[~agree][["subreddit", "sentiment", "scorer_label", "human_label", "text"]])

Labelled: 45 of 45 by Labeller A, Labeller B, Labeller C
Agreement (scorer sign at ±0.05 vs human): 37.8% (17/45)
Per scorer class: {'bearish': 0.07, 'bullish': 0.33, 'neutral': 0.73}
Per subreddit (tiny n, indicative only):
            labelled  agreement
subreddit                      
Gold              20       0.40
Silverbugs        24       0.38
investing          1       0.00
Without r/Silverbugs: 38.1% (8/21)
Human label share by subreddit:
human_label  bearish  bullish  neutral
subreddit                             
Gold            0.10     0.30     0.60
Silverbugs      0.08     0.29     0.62
investing       0.00     0.00     1.00


,scorer bearish,scorer neutral,scorer bullish
human bearish,1,1,2
human neutral,9,11,8
human bullish,5,3,5



Disagreements (look for missing lexicon terms or sarcasm):


,subreddit,sentiment,scorer_label,human_label,text
1,Silverbugs,0.8442,bullish,neutral,A Nice Way to Wind Down Enjoying a good book w...
2,Silverbugs,0.7960,bullish,neutral,Got her this week I forgot that I ordered this...
6,Silverbugs,-0.7867,bearish,neutral,Ebay question I rarely use eBay and I don't un...
7,Gold,-0.4019,bearish,neutral,"The Royal Mint , sell back Has anyone had a pr..."
8,Silverbugs,-0.6369,bearish,neutral,Double Die 1928P Peace Is there even such a th...
9,Gold,-0.4215,bearish,bullish,Are Gold coin quick commerce app legit? Today ...
10,Silverbugs,-0.5346,bearish,neutral,My First(Second) Stack Had to restart the stac...
11,investing,0.9083,bullish,neutral,I tracked a California tax auction from the co...
13,Silverbugs,0.7425,bullish,neutral,Mail call Finally was able to finish off my co...
15,Silverbugs,-0.5255,bearish,bullish,State of the stack Had an oz since my baptism ...


**Lexicon follow-up.** If the disagreements show an obvious missing finance term, the finding is
written up as *"adding X to `FINANCE_LEXICON` would fix N of the disagreements"*. With collection
frozen, the database is **not** re-scored, so every stored sentiment value stays as it is.

## 4. Second scorer: FinBERT, side by side with VADER

VADER measures emotional tone, not a view on the gold price. **FinBERT** (`ProsusAI/finbert`, pinned
revision, see `sentiment/finbert.py`) is a free, ready-made classifier trained on financial news. It
is used **as is**: no fine-tuning and no thresholds fitted to these labels (they are the exam, so
tuning on them would inflate the score). Its label is simply its most likely class
(positive → bullish, negative → bearish). Scores live in `data/clean/post_sentiment_finbert.parquet`;
the database and this CSV are only read.

Both scorers are compared on **the same labelled rows**. The exact McNemar test asks whether one is
right more often than the other by more than chance. Most hand-labelled posts are neutral, so raw
agreement flatters a scorer that says "neutral" to everything. The table therefore also shows that
**always-neutral yardstick** and a **balanced** score (mean agreement per human class).
**Pre-commitment:** the scorer that agrees better with the human labels *and* beats the yardstick is
the recommended one. This is decided on label agreement, never on
model accuracy. Per-post FinBERT labels for the sample are shown only once all 45 rows are labelled,
so they can't influence the remaining labelling.

In [7]:
from scipy.stats import binomtest

fin = pd.read_parquet("data/clean/post_sentiment_finbert.parquet")
assert fin["post_id"].is_unique and fin.notna().all().all()
print("FinBERT label share, all de-duplicated posts in window:")
print(posts.merge(fin, on="post_id")["finbert_label"].value_counts(normalize=True).round(3).to_string())

if done.empty:
    print("Awaiting human labels.")
else:
    both = done.merge(fin[["post_id", "finbert_score", "finbert_label"]], on="post_id", validate="1:1")
    v_ok, f_ok = both["human_label"] == both["scorer_label"], both["human_label"] == both["finbert_label"]
    only_v, only_f = int((v_ok & ~f_ok).sum()), int((f_ok & ~v_ok).sum())
    p = binomtest(only_f, only_v + only_f, 0.5).pvalue if only_v + only_f else 1.0
    no_sb = both["subreddit"] != "Silverbugs"
    n_ok = both["human_label"] == "neutral"  # yardstick: a "scorer" that always says neutral
    by_class = lambda ok: ok.groupby(both["human_label"]).mean().mean()  # balanced: mean agreement per human class
    print(f"\nLabelled rows: {len(both)} of {len(val)}" + ("" if len(both) == len(val) else "  (PREVIEW: labelling not finished)"))
    display(pd.DataFrame({
        "agreement": [v_ok.mean(), f_ok.mean()],
        "right / labelled": [f"{v_ok.sum()}/{len(both)}", f"{f_ok.sum()}/{len(both)}"],
        "without r/Silverbugs": [v_ok[no_sb].mean(), f_ok[no_sb].mean()],
        "balanced (mean per human class)": [by_class(v_ok), by_class(f_ok)],
    }, index=["VADER (±0.05 band)", "FinBERT (most likely class)"]).round(3))
    print(f"Yardstick, always 'neutral': agreement {n_ok.mean():.3f} ({n_ok.sum()}/{len(both)}), "
          f"balanced {1 / both['human_label'].nunique():.3f}. A useful scorer must beat this.")
    print(f"only VADER right: {only_v}   only FinBERT right: {only_f}   exact McNemar p = {p:.3f}")
    print("Agreement by HUMAN label (how well each scorer finds each kind of post):")
    print(pd.DataFrame({"n": both.groupby("human_label").size(),
                        "VADER": v_ok.groupby(both["human_label"]).mean(),
                        "FinBERT": f_ok.groupby(both["human_label"]).mean()}).round(2).to_string())
    cm_f = pd.DataFrame(confusion_matrix(both["human_label"], both["finbert_label"], labels=labels),
                        index=[f"human {l}" for l in labels], columns=[f"FinBERT {l}" for l in labels])
    display(cm_f)
    if len(both) == len(val):
        print("\nFinBERT disagreements:")
        display(both[~f_ok][["subreddit", "finbert_score", "finbert_label", "human_label", "text"]])

FinBERT label share, all de-duplicated posts in window:
finbert_label
neutral    0.959
bullish    0.021
bearish    0.021

Labelled rows: 45 of 45


,agreement,right / labelled,without r/Silverbugs,balanced (mean per human class)
VADER (±0.05 band),0.378,17/45,0.381,0.342
FinBERT (most likely class),0.600,27/45,0.619,0.335


Yardstick, always 'neutral': agreement 0.622 (28/45), balanced 0.333. A useful scorer must beat this.
only VADER right: 7   only FinBERT right: 17   exact McNemar p = 0.064
Agreement by HUMAN label (how well each scorer finds each kind of post):
              n  VADER  FinBERT
human_label                    
bearish       4   0.25     0.00
bullish      13   0.38     0.08
neutral      28   0.39     0.93


,FinBERT bearish,FinBERT neutral,FinBERT bullish
human bearish,0,4,0
human neutral,1,26,1
human bullish,0,12,1



FinBERT disagreements:


,subreddit,finbert_score,finbert_label,human_label,text
7,Gold,-0.690225,bearish,neutral,"The Royal Mint , sell back Has anyone had a pr..."
9,Gold,-0.017972,neutral,bullish,Are Gold coin quick commerce app legit? Today ...
14,Silverbugs,0.167114,neutral,bullish,New Perth Mint 10oz Bar….in love Morning fello...
15,Silverbugs,0.054492,neutral,bullish,State of the stack Had an oz since my baptism ...
16,Silverbugs,0.084114,neutral,bearish,Traded up my constitutional silver for some Am...
20,Gold,-0.061585,neutral,bullish,who do i sell gold to? [removed]
22,Gold,0.003094,neutral,bullish,Any opinions on if this looks like real gold M...
23,Silverbugs,-0.000513,neutral,bullish,"I bought my first coin, 1987 ASE from JM bulli..."
24,Silverbugs,0.149071,neutral,bullish,"New Drop First time seeing this, thought I sho..."
25,Silverbugs,0.008497,neutral,bullish,It’s kinda sad to look at this now i consolida...
